# Implementación de Controles de Seguridad (Mid)

Objetivos:
- Implementar controles de acceso (RBAC, ABAC) en sistemas de datos.
- Aplicar encriptación at-rest y in-transit.
- Configurar auditoría y logging de accesos.
- Diseñar estrategias de anonimización y masking dinámico.

<!-- course-nav -->
> Ruta: [Inicio del repositorio](../README.md) | [Índice del nivel](README.md)  
> Anterior: [Metadatos, Catálogo y Linaje de Datos (Mid)](05_metadatos_catalogo_linaje.ipynb) | Siguiente: [Optimización, Dimensionamiento y FinOps de Datos](07_optimizacion_dimensionamiento_finops.ipynb)


<!-- course-quality-structure -->
## Prerrequisitos

- Haber revisado los notebooks anteriores de la ruta o contar con experiencia equivalente.
- Conocer los conceptos básicos mencionados en el README del nivel.
- Tener disponible el repositorio local y las dependencias de `requirements.txt` cuando se ejecuten celdas de código.

## Preparación

1. Trabaja desde la raíz del repositorio.
2. Activa tu entorno de Python.
3. Instala dependencias con `pip install -r requirements.txt` si aún no lo hiciste.
4. Ejecuta las celdas en orden para conservar el contexto de variables y resultados.

## Mapa del notebook

1. 1. Control de Acceso: RBAC vs ABAC
2. 2. Encriptación de Datos
3. 3. Auditoría y Logging
4. 4. Anonimización y Masking Dinámico
5. 5. Ejercicio Integrador
6. 6. Resumen

## Diagrama de continuidad

```mermaid
flowchart LR
    A["Notebook anterior"] --> B["Implementación de Controles de Seguridad (Mid)"] --> C["Notebook siguiente"]
```


## 1. Control de Acceso: RBAC vs ABAC

### RBAC (Role-Based Access Control)
**Concepto:** Permisos asignados a roles; usuarios heredan permisos del rol.

**Ejemplo:**
```
Roles:
- analista_ventas: SELECT en silver.ventas, gold.kpi_ventas
- admin_datos: ALL en bronze.*, silver.*, gold.*
- usuario_externo: SELECT en gold.reportes_publicos
```

**Ventajas:** Simple, escalable para equipos grandes.
**Limitaciones:** No considera contexto (hora, ubicación, atributos del recurso).

### ABAC (Attribute-Based Access Control)
**Concepto:** Permisos basados en evaluación de atributos (usuario, recurso, contexto).

**Ejemplo de regla:**
```python
if (user.department == 'Finanzas' AND 
    resource.classification == 'Confidencial' AND 
    context.time in business_hours AND
    context.location == 'office_network'):
    allow_access()
```

**Ventajas:** Granularidad fina, adaptable a contextos complejos.
**Limitaciones:** Más complejo de configurar y auditar.

In [ ]:
# Simulación simple de RBAC en Python
import pandas as pd

# Roles y permisos
roles_permisos = {
    'analista_ventas': {'schemas': ['silver', 'gold'], 'operations': ['SELECT']},
    'admin_datos': {'schemas': ['bronze', 'silver', 'gold'], 'operations': ['SELECT', 'INSERT', 'UPDATE', 'DELETE']},
    'usuario_externo': {'schemas': ['gold'], 'operations': ['SELECT'], 'tables': ['reportes_publicos']}
}

# Usuario y solicitud de acceso
usuario = {'nombre': 'Ana', 'rol': 'analista_ventas'}
solicitud = {'schema': 'silver', 'tabla': 'ventas', 'operacion': 'SELECT'}

def verificar_acceso_rbac(user, request, roles):
    permisos = roles.get(user['rol'])
    if not permisos:
        return False
    if request['schema'] not in permisos['schemas']:
        return False
    if request['operacion'] not in permisos['operations']:
        return False
    # Verificar tabla específica si aplica
    if 'tables' in permisos and request['tabla'] not in permisos['tables']:
        return False
    return True

acceso = verificar_acceso_rbac(usuario, solicitud, roles_permisos)
print(f"Usuario: {usuario['nombre']} ({usuario['rol']})")
print(f"Solicitud: {solicitud['operacion']} en {solicitud['schema']}.{solicitud['tabla']}")
print(f"Acceso: {'✓ PERMITIDO' if acceso else '✗ DENEGADO'}")

## 2. Encriptación de Datos

### 2.1. At-Rest (En Reposo)
**Dónde:** Archivos en disco, tablas en BD, objetos en S3/Blob Storage.

**Métodos:**
- **Cifrado transparente (TDE):** BD cifra automáticamente (SQL Server, PostgreSQL).
- **Cifrado a nivel storage:** S3 SSE (Server-Side Encryption), Azure Storage Encryption.
- **KMS (Key Management Service):** Gestión centralizada de claves (AWS KMS, Azure Key Vault, GCP KMS).

**Ejemplo configuración S3:**
```python
# boto3 (AWS SDK)
s3.put_object(
    Bucket='my-datalake',
    Key='bronze/ventas/2025-11-07.parquet',
    Body=data,
    ServerSideEncryption='aws:kms',
    SSEKMSKeyId='arn:aws:kms:region:account:key/xxx'
)
```

### 2.2. In-Transit (En Tránsito)
**Dónde:** Conexiones cliente→servidor, microservicios, ETL entre sistemas.

**Protocolos:**
- **TLS 1.2/1.3** (Transport Layer Security): HTTPS, conexiones DB.
- **mTLS** (mutual TLS): Autenticación bidireccional (microservicios).

**Validación:**
- Certificados SSL/TLS válidos.
- Sin downgrades a versiones inseguras (SSLv3, TLS 1.0).

**Ejemplo conexión segura:**
```python
import psycopg2
conn = psycopg2.connect(
    host='db.example.com',
    sslmode='require',  # Forzar TLS
    sslrootcert='/path/to/ca-cert.pem'
)
```

In [ ]:
# Demostración: cifrado simétrico básico con cryptography
from cryptography.fernet import Fernet

# Generar clave (en prod: usar KMS)
clave = Fernet.generate_key()
cipher = Fernet(clave)

# Cifrar dato sensible
email_original = 'ana.garcia@example.com'
email_cifrado = cipher.encrypt(email_original.encode())
print(f'Email cifrado: {email_cifrado}')

# Descifrar (solo con clave correcta)
email_descifrado = cipher.decrypt(email_cifrado).decode()
print(f'Email descifrado: {email_descifrado}')

print('\n⚠️ En producción: almacenar clave en KMS, NO en código.')

## 3. Auditoría y Logging

### 3.1. ¿Qué Auditar?
- **Accesos a datos sensibles:** Quién, cuándo, qué tabla/registro.
- **Cambios de permisos:** Rol otorgado/revocado.
- **Cambios de esquema:** ALTER TABLE, DROP.
- **Exportaciones masivas:** SELECT * con > 10K filas.
- **Fallos de autenticación:** Intentos fallidos.

### 3.2. Formato de Log
```json
{
  "timestamp": "2025-11-07T14:30:15Z",
  "user": "ana.garcia@empresa.com",
  "action": "SELECT",
  "resource": "gold.clientes_perfil_360",
  "rows_returned": 1250,
  "client_ip": "10.20.30.40",
  "result": "success"
}
```

### 3.3. Retención y Protección
- Logs inmutables (write-once, append-only).
- Retención: mínimo 90 días (compliance: 1-7 años).
- Almacenamiento seguro: cifrado, acceso restringido.

In [ ]:
# Simulación de logging de acceso
import json
from datetime import datetime, timezone

def log_acceso(user, action, resource, rows=None, ip='unknown', result='success'):
    log_entry = {
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'user': user,
        'action': action,
        'resource': resource,
        'rows_returned': rows,
        'client_ip': ip,
        'result': result
    }
    # En prod: enviar a sistema centralizado (CloudWatch, Splunk, ELK)
    print(json.dumps(log_entry, indent=2))

# Ejemplo: analista consulta ventas
log_acceso(
    user='ana.garcia@empresa.com',
    action='SELECT',
    resource='silver.ventas_unificadas',
    rows=3420,
    ip='10.20.30.40'
)

# Ejemplo: intento denegado
log_acceso(
    user='externo@partner.com',
    action='SELECT',
    resource='bronze.transacciones_raw',
    ip='192.168.1.100',
    result='access_denied'
)

## 4. Anonimización y Masking Dinámico

### 4.1. Anonimización (Irreversible)
**Técnicas:**
- **Supresión:** Eliminar columna PII.
- **Generalización:** `edad=28` → `rango_edad='25-30'`.
- **Agregación:** Solo estadísticas (no registros individuales).

### 4.2. Pseudonimización (Reversible con clave)
**Técnicas:**
- **Hashing:** SHA-256(email) → identificador único, pero no reversible sin rainbow tables.
- **Tokenización:** Reemplazar valor real con token; mapeo en bóveda segura.

### 4.3. Dynamic Data Masking (DDM)
**Concepto:** Mostrar datos enmascarados según permisos del usuario en tiempo real.

**Ejemplo:**
```sql
-- Usuario con permiso parcial ve:
SELECT email FROM clientes;
-- Resultado: a***@example.com, l***@example.com

-- Usuario con permiso completo ve:
-- Resultado: ana@example.com, luis@example.com
```

**Plataformas:** Snowflake Masking Policies, BigQuery Column-level Security, Azure SQL DDM.

In [ ]:
# Ejemplo: masking dinámico según rol
import pandas as pd

df_clientes = pd.DataFrame({
    'cliente_id': [1, 2, 3],
    'nombre': ['Ana García', 'Luis Pérez', 'María López'],
    'email': ['ana@example.com', 'luis@example.com', 'maria@example.com'],
    'saldo': [1500, 2300, 800]
});

def aplicar_masking(df, user_role):
    df_masked = df.copy()
    if user_role in ['analista_externo', 'partner']:
        # Mask: mostrar solo inicial + ***
        df_masked['nombre'] = df_masked['nombre'].apply(lambda x: x[0] + '***')
        df_masked['email'] = df_masked['email'].apply(lambda x: x[0] + '***@' + x.split('@')[1])
    elif user_role == 'analista_interno':
        # Partial mask: email completo, nombre parcial
        df_masked['nombre'] = df_masked['nombre'].apply(lambda x: x.split()[0] + ' ***')
    # admin_datos: sin masking
    return df_masked

print('Vista: analista_externo')
print(aplicar_masking(df_clientes, 'analista_externo'))

print('\nVista: admin_datos')
print(aplicar_masking(df_clientes, 'admin_datos'))

## 5. Ejercicio Integrador

**Escenario:** Diseñar controles de seguridad para `gold.clientes_perfil_360` (contiene: cliente_id, nombre, email, teléfono, historial_compras, score_credito).

**Tareas:**
1. Define 3 roles con permisos diferenciados (RBAC).
2. Especifica qué campos aplicar dynamic masking para cada rol.
3. Configura auditoría: ¿qué eventos loguear?
4. Propón estrategia de encriptación (at-rest y in-transit).
5. Diseña proceso de acceso para un partner externo (workflow + controles).

In [ ]:
# Plantilla solución
solucion = {
    'roles_rbac': {
        'analista_interno': {'permisos': 'SELECT', 'masking': 'teléfono parcial'},
        'admin_finanzas': {'permisos': 'SELECT', 'masking': 'sin masking'},
        'partner_externo': {'permisos': 'SELECT agregado', 'masking': 'nombre/email/teléfono completo'}
    },
    'auditoria': ['SELECT con >100 filas', 'EXPORT', 'cambios de permisos'],
    'encriptacion': {
        'at_rest': 'TDE en BD + S3 SSE-KMS',
        'in_transit': 'TLS 1.3 obligatorio'
    },
    'workflow_partner': [
        '1. Solicitud con justificación',
        '2. Aprobación Owner + Legal',
        '3. Provisión rol con masking completo',
        '4. Acceso limitado a vista agregada (sin registros individuales)',
        '5. Auditoría de todas las consultas',
        '6. Revisión trimestral'
    ]
}
import json; print(json.dumps(solucion, indent=2, ensure_ascii=False))

## 6. Resumen

- **Control de acceso:** RBAC (simple, escalable) vs ABAC (granular, contextual).
- **Encriptación:** At-rest (TDE, KMS) y in-transit (TLS/mTLS) para proteger datos.
- **Auditoría:** Loguear accesos, cambios, exportaciones; logs inmutables con retención compliance.
- **Masking:** Anonimización (irreversible), pseudonimización (reversible), dynamic masking (según rol).
- **Implementación:** Balancear seguridad con usabilidad; automatizar controles donde sea posible.

<!-- course-quality-close -->
## Ejercicio de cierre

Revisa el caso trabajado en el notebook y documenta una decisión de arquitectura: contexto, opción elegida, alternativas descartadas, riesgos y métricas de validación.

## Evaluación

- El notebook se ejecuta en orden sin depender de estado oculto.
- Las decisiones técnicas están justificadas con trade-offs claros.
- Los resultados, diagramas o tablas apoyan la explicación y no reemplazan el razonamiento.
- Las referencias usadas son oficiales, primarias o están incluidas en la bibliografía del curso.

## Referencias

- OWASP API Security Project: https://owasp.org/www-project-api-security/
- Referencias del curso: ../04-recursos/referencias.md
